# NPA screening of the four class-29 perturbation facets

## Purpose

The free-$\beta$ perturbation searches with $\epsilon=10^{-3}$ and
$\epsilon=10^{-2}$ returned the same four exact class-29 facets. This notebook
embeds those four inequalities directly and screens each one with the NPA
hierarchy at levels 2, 3, and 4.

For every inequality in the exact normalization produced by the search

$$
I(p)=c\cdot p\geq\beta,
$$

the notebook reports separately:

- the direct NPA minimum $L_k=\min_{p\in\mathcal Q_k}I(p)$;
- the siglocal bound $\beta$;
- the direct slack $\Delta_k=L_k-\beta$ and its white-noise-normalized value;
- the optimistic white-noise tolerance allowed by the NPA relaxation.

The four facets are identified by their perturbation provenance:

- the unperturbed class-29 separator;
- ordered pair $(1,3)$;
- ordered pair $(1,585)$;
- ordered pair $(1,849)$.

The notebook is self-contained. No TSV input file is required.


## Interpretation of the NPA output

The NPA set $\mathcal Q_k$ is an outer approximation of the quantum set. For a
minimization problem, $L_k$ is therefore a lower bound on the true quantum
minimum.

- If $L_k\geq\beta$ beyond numerical tolerance, level $k$ excludes a quantum
  violation.
- If $L_k<\beta$, the relaxation permits a violation. This identifies a
  promising inequality, but it does not provide a quantum state and
  measurements.
- A see-saw calculation is required afterward to produce an explicit quantum
  realization.

SCS is a numerical first-order solver. The returned values are numerical NPA
bounds, not formally certified SDP bounds. Convergence across levels 2, 3,
and 4 is printed as a numerical diagnostic.


## How to run

1. Open this notebook with the Julia kernel.
2. Choose **Kernel → Restart Kernel and Run All Cells**.
3. The package-bootstrap cell installs only packages missing from the active
   Julia environment.

The NPA scan is checkpointed after every completed SDP. Results are written to
`cycle_class29_perturbed_npa_results` beside the notebook:

- `class29_npa_all_levels.tsv`;
- `class29_npa_level4_ranking.tsv`;
- `npa_checkpoint.jls`;
- `run_metadata.txt`.

If the kernel is interrupted, rerunning all cells resumes completed
candidate/level combinations from the checkpoint.


In [4]:
# Package bootstrap. Existing packages are not upgraded or modified; only
# packages absent from the active Julia environment are installed.
import Pkg

required_packages = [
    "JuMP",
    "MathOptInterface",
    "QuantumNPA",
    "SCS",
]

missing_packages = [
    package for package in required_packages
    if Base.find_package(package) === nothing
]

if !isempty(missing_packages)
    println("Installing missing Julia packages: ", join(missing_packages, ", "))
    Pkg.add(missing_packages)
else
    println("All required Julia packages are already available.")
end


All required Julia packages are already available.


In [5]:
using Dates
using JuMP
using LinearAlgebra
using Printf
using QuantumNPA
using SCS
using Serialization
using SparseArrays

# Import MathOptInterface directly. In particular, do not use
# `JuMP.MathOptInterface`, which is incompatible with the user's Julia 1.12
# installation.
import MathOptInterface as MOI

# -------------------------- USER CONFIGURATION --------------------------

NPA_LEVELS = [2, 3, 4]

# Tight SCS settings already used successfully in the preceding NPA notebooks.
SCS_EPS_ABS = 1e-8
SCS_EPS_REL = 1e-8
SCS_MAX_ITERS = 1_000_000

# These tolerances classify numerical outputs and diagnose monotonicity. They
# do not change the SDP or any inequality.
NPA_SCREENING_TOLERANCE = 5e-4
NPA_MONOTONICITY_TOLERANCE = 5e-4

RESULTS_DIRECTORY = joinpath(
    pwd(),
    "cycle_class29_perturbed_npa_results",
)
RESUME_NPA_SCAN = true
STOP_ON_NPA_ERROR = true

# ------------------------------------------------------------------------

TIGHT_SCS_SOLVER = optimizer_with_attributes(
    SCS.Optimizer,
    "eps_abs" => SCS_EPS_ABS,
    "eps_rel" => SCS_EPS_REL,
    "max_iters" => SCS_MAX_ITERS,
    "verbose" => 0,
)

const TRIPLES = [(u, v, w) for u in 0:1 for v in 0:1 for w in 0:1]

mkpath(RESULTS_DIRECTORY)
println("NPA output directory: ", abspath(RESULTS_DIRECTORY))


NPA output directory: C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_project\CV_dropbox_content\Projeto_Ze_Gabrielzinho_Rabelo_Planeta\Jupiter\cycle_class29_perturbed_npa_results


## 1. Canonical probability coordinates

A behavior is stored as a 64-vector. Input blocks use the order

`xyz = 000, 001, 010, 011, 100, 101, 110, 111`,

and each block uses the output order

`abc = 000, 001, 010, 011, 100, 101, 110, 111`.

Thus $z$ and $c$ are the fastest-changing input and output bits. The same map
is used for deterministic vertices, the class-29 target, coefficients, and
NPA probability operators.


In [7]:
"""One-based index of p(a,b,c|x,y,z) in the canonical 64-vector."""
prob_index(a, b, c, x, y, z) =
    1 + c + 2*b + 4*a + 8*z + 16*y + 32*x

@assert [prob_index(a, b, c, 0, 0, 0) for (a, b, c) in TRIPLES] == collect(1:8)
@assert [prob_index(0, 0, 0, x, y, z) for (x, y, z) in TRIPLES] == collect(1:8:57)


## 2. Correct siglocal vertices

Each party uses a deterministic binary function of two binary inputs. The two
communication orientations are

| orientation | Alice | Bob | Charlie |
|---|---|---|---|
| clockwise | $a=f_A(x,z)$ | $b=f_B(y,x)$ | $c=f_C(z,y)$ |
| counterclockwise | $a=f_A(x,y)$ | $b=f_B(y,z)$ | $c=f_C(z,x)$ |

Each orientation contains $16^3=4096$ deterministic behaviors. Their
intersection consists of the 64 fully local deterministic behaviors, so their
union contains 8128 distinct siglocal vertices and has affine dimension 50.


In [9]:
"""Evaluate one of the 16 deterministic functions of two binary inputs."""
function response(mask::Int, u::Int, v::Int)
    @assert 0 <= mask <= 15
    @assert u in 0:1 && v in 0:1
    return Int((mask >> (2*u + v)) & 1)
end

"""Construct one deterministic behavior in canonical coordinates."""
function deterministic_vertex(
    fA::Int,
    fB::Int,
    fC::Int,
    orientation::Symbol,
)
    vertex = zeros(Int8, 64)

    for (x, y, z) in TRIPLES
        if orientation === :clockwise
            a = response(fA, x, z)
            b = response(fB, y, x)
            c = response(fC, z, y)
        elseif orientation === :counterclockwise
            a = response(fA, x, y)
            b = response(fB, y, z)
            c = response(fC, z, x)
        else
            error("Unknown orientation: $orientation")
        end

        vertex[prob_index(a, b, c, x, y, z)] = 1
    end

    return vertex
end

function generate_orientation(orientation::Symbol)
    vertices = Matrix{Int8}(undef, 16^3, 64)
    row = 1
    for fA in 0:15, fB in 0:15, fC in 0:15
        vertices[row, :] .= deterministic_vertex(fA, fB, fC, orientation)
        row += 1
    end
    return vertices
end

row_set(vertices) = Set(Tuple(row) for row in eachrow(vertices))

function numerical_affine_dimension(points; atol=1e-9)
    number_of_points = size(points, 1)
    number_of_points == 0 && return -1
    number_of_points == 1 && return 0
    differences = Float64.(points[2:end, :]) .- Float64.(points[1:1, :])
    return rank(differences; atol=atol, rtol=0.0)
end

function validate_deterministic_vertices(vertices)
    @assert size(vertices, 2) == 64
    for vertex in eachrow(vertices)
        @assert all(value -> value == 0 || value == 1, vertex)
        for context in 0:7
            block = (8*context + 1):(8*context + 8)
            @assert sum(vertex[block]) == 1
        end
    end
    return true
end

vertices_clockwise = generate_orientation(:clockwise)
vertices_counterclockwise = generate_orientation(:counterclockwise)
vertices_cycle = unique(
    vcat(vertices_clockwise, vertices_counterclockwise);
    dims=1,
)

clockwise_set = row_set(vertices_clockwise)
counterclockwise_set = row_set(vertices_counterclockwise)

validate_deterministic_vertices(vertices_clockwise)
validate_deterministic_vertices(vertices_counterclockwise)
validate_deterministic_vertices(vertices_cycle)

@assert size(vertices_clockwise, 1) == 4096
@assert size(vertices_counterclockwise, 1) == 4096
@assert length(intersect(clockwise_set, counterclockwise_set)) == 64
@assert size(vertices_cycle, 1) == 8128
@assert numerical_affine_dimension(vertices_clockwise) == 44
@assert numerical_affine_dimension(vertices_counterclockwise) == 44
@assert numerical_affine_dimension(vertices_cycle) == 50

println("Corrected siglocal polytope")
println("  clockwise vertices:        ", size(vertices_clockwise, 1))
println("  counterclockwise vertices: ", size(vertices_counterclockwise, 1))
println("  intersection:              ", length(intersect(clockwise_set, counterclockwise_set)))
println("  distinct union:            ", size(vertices_cycle, 1))
println("  affine dimension:          ", numerical_affine_dimension(vertices_cycle))


Corrected siglocal polytope
  clockwise vertices:        4096
  counterclockwise vertices: 4096
  intersection:              64
  distinct union:            8128
  affine dimension:          50


## 3. Exact affine-rank certification

Facetness requires the affine dimension of the saturating vertices. The next
cell computes integer-matrix ranks modulo several primes and uses a Hadamard
bound to certify that the modular result equals the rational rank. This gives
an exact face-dimension test, independent of floating-point singular-value
thresholds.


In [11]:
const RANK_PRIMES = Int64[
    1_000_000_007,
    1_000_000_009,
    998_244_353,
    1_004_535_809,
]

mod_multiply(a::Int64, b::Int64, modulus::Int64) =
    Int64(mod(Int128(a) * Int128(b), Int128(modulus)))

function mod_power(base::Int64, exponent::Int64, modulus::Int64)
    result = Int64(1)
    factor = mod(base, modulus)
    power = exponent
    while power > 0
        if isodd(power)
            result = mod_multiply(result, factor, modulus)
        end
        factor = mod_multiply(factor, factor, modulus)
        power >>= 1
    end
    return result
end

function rank_mod_prime(matrix::AbstractMatrix{<:Integer}, modulus::Int64)
    n_rows, n_columns = size(matrix)
    work = Matrix{Int64}(undef, n_rows, n_columns)
    @inbounds for index in eachindex(matrix)
        work[index] = mod(Int64(matrix[index]), modulus)
    end

    pivot_row = 1
    @inbounds for column in 1:n_columns
        pivot_row > n_rows && break

        pivot = 0
        for row in pivot_row:n_rows
            if work[row, column] != 0
                pivot = row
                break
            end
        end
        pivot == 0 && continue

        if pivot != pivot_row
            for j in 1:n_columns
                work[pivot_row, j], work[pivot, j] =
                    work[pivot, j], work[pivot_row, j]
            end
        end

        inverse_pivot = mod_power(
            work[pivot_row, column],
            modulus - 2,
            modulus,
        )

        for row in (pivot_row + 1):n_rows
            entry = work[row, column]
            entry == 0 && continue
            factor = mod_multiply(entry, inverse_pivot, modulus)
            for j in column:n_columns
                product = mod_multiply(factor, work[pivot_row, j], modulus)
                work[row, j] = Int64(mod(
                    Int128(work[row, j]) - Int128(product),
                    Int128(modulus),
                ))
            end
        end
        pivot_row += 1
    end
    return pivot_row - 1
end

function certified_integer_rank(matrix::AbstractMatrix{<:Integer})
    n_rows, n_columns = size(matrix)
    min(n_rows, n_columns) == 0 && return 0

    ranks = [rank_mod_prime(matrix, prime) for prime in RANK_PRIMES]
    certified_rank = maximum(ranks)
    certified_rank == min(n_rows, n_columns) && return certified_rank

    maximum_squared_row_norm = maximum(
        sum(BigInt(matrix[row, column])^2 for column in 1:n_columns)
        for row in 1:n_rows
    )
    maximum_squared_row_norm == 0 && return 0

    next_minor_size = certified_rank + 1
    modulus_product = prod(BigInt(prime) for prime in RANK_PRIMES)
    hadamard_squared_bound = maximum_squared_row_norm^next_minor_size

    modulus_product^2 > hadamard_squared_bound || error(
        "The modular primes are insufficient to certify this rank exactly.",
    )
    return certified_rank
end

function certified_affine_dimension(points::AbstractMatrix{<:Integer})
    n_points, dimension = size(points)
    n_points == 0 && return -1
    n_points == 1 && return 0

    differences = Matrix{Int64}(undef, n_points - 1, dimension)
    @inbounds for row in 2:n_points, column in 1:dimension
        differences[row - 1, column] =
            Int64(points[row, column]) - Int64(points[1, column])
    end
    return certified_integer_rank(differences)
end

const SIGLOCAL_AFFINE_DIMENSION = certified_affine_dimension(vertices_cycle)
@assert SIGLOCAL_AFFINE_DIMENSION == 50
println("Exact siglocal affine dimension: ", SIGLOCAL_AFFINE_DIMENSION)


Exact siglocal affine dimension: 50


## 4. Class-29 target and the four embedded facets

The class-29 representative is reconstructed exactly from its 26 correlators.
Each facet is stored both in the exact rational normalization produced by the
search and in primitive integer form. No floating-point coefficient is read.

The two perturbation searches returned exactly the same four primitive
coefficient vectors. The value of $\epsilon$ therefore affects their discovery
history but not the NPA problems solved below.


In [13]:
const CLASS29_CORRELATORS = Rational{Int}[
    1//3, 1//3, 1//3, 1//3, 1//3, 1//3,
    1, -1//3, -1//3, -1//3,
    1, -1//3, -1//3, -1//3,
    1, -1//3, -1//3, -1//3,
    1//3, 1//3, 1//3, -1, 1//3, -1, -1, -1,
]

outcome_sign(bit::Int) = bit == 0 ? 1 : -1

function correlators_to_probabilities(correlators)
    @assert length(correlators) == 26
    q = Rational{Int}.(correlators)
    probabilities = fill(0//1, 64)

    A = q[1:2]
    B = q[3:4]
    C = q[5:6]
    AB = q[7:10]
    AC = q[11:14]
    BC = q[15:18]
    ABC = q[19:26]

    for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
        sa = outcome_sign(a)
        sb = outcome_sign(b)
        sc = outcome_sign(c)
        probabilities[prob_index(a, b, c, x, y, z)] = (
            1 +
            sa*A[x + 1] + sb*B[y + 1] + sc*C[z + 1] +
            sa*sb*AB[2*x + y + 1] +
            sa*sc*AC[2*x + z + 1] +
            sb*sc*BC[2*y + z + 1] +
            sa*sb*sc*ABC[4*x + 2*y + z + 1]
        ) / 8
    end
    return probabilities
end

const CLASS29_TARGET = correlators_to_probabilities(CLASS29_CORRELATORS)
const UNIFORM_BEHAVIOUR = fill(BigInt(1)//BigInt(8), 64)

const RAW_CANDIDATES = [
    (
        candidate_id = "F29_unperturbed",
        source = :unperturbed,
        pair_j = 0,
        pair_k = 0,
        beta = -28//15,
        coefficients = Rational{BigInt}[
            -8//45,4//45,-4//15,0,-4//15,0,-16//45,-28//45,-4//15,-4//15,16//45,0,16//45,0,4//45,4//45,-4//15,0,0,-4//45,16//45,-4//15,4//15,0,4//15,4//15,0,0,0,8//15,4//15,4//15,0,4//15,28//45,0,-4//15,-16//45,0,-4//15,-8//15,-8//15,-4//5,-4//15,-4//15,-4//15,0,0,-8//9,-4//5,-52//45,-8//15,-28//45,0,-8//9,-4//15,8//15,0,4//15,4//15,-4//15,-4//15,0,0
        ],
        primitive_beta = BigInt(-21),
        primitive_coefficients = BigInt[
            -2,1,-3,0,-3,0,-4,-7,-3,-3,4,0,4,0,1,1,-3,0,0,-1,4,-3,3,0,3,3,0,0,0,6,3,3,0,3,7,0,-3,-4,0,-3,-6,-6,-9,-3,-3,-3,0,0,-10,-9,-13,-6,-7,0,-10,-3,6,0,3,3,-3,-3,0,0
        ],
        saturation_count_from_search = 270,
    ),
    (
        candidate_id = "F29_pair_1_3",
        source = :ordered_pair,
        pair_j = 1,
        pair_k = 3,
        beta = -28//15,
        coefficients = Rational{BigInt}[
            -8//15,-4//15,-4//15,0,-4//15,0,0,-4//15,-4//15,-4//15,0,0,0,0,-4//15,-4//15,-4//15,0,0,4//15,0,-4//15,4//15,0,4//15,4//15,0,0,0,8//15,4//15,4//15,0,4//15,4//15,0,-4//15,0,0,-4//15,-8//15,-8//15,-4//5,-4//15,-4//15,-4//15,0,0,-8//15,-4//5,-4//5,-8//15,-4//15,0,-8//15,-4//15,8//15,0,4//15,4//15,-4//15,-4//15,0,0
        ],
        primitive_beta = BigInt(-7),
        primitive_coefficients = BigInt[
            -2,-1,-1,0,-1,0,0,-1,-1,-1,0,0,0,0,-1,-1,-1,0,0,1,0,-1,1,0,1,1,0,0,0,2,1,1,0,1,1,0,-1,0,0,-1,-2,-2,-3,-1,-1,-1,0,0,-2,-3,-3,-2,-1,0,-2,-1,2,0,1,1,-1,-1,0,0
        ],
        saturation_count_from_search = 322,
    ),
    (
        candidate_id = "F29_pair_1_585",
        source = :ordered_pair,
        pair_j = 1,
        pair_k = 585,
        beta = -28//15,
        coefficients = Rational{BigInt}[
            -8//15,-4//15,-4//15,0,-4//15,0,0,-4//15,-4//15,-4//15,0,0,0,0,-4//15,4//15,-4//15,0,0,4//15,0,-4//15,4//15,8//15,4//15,4//15,0,0,0,8//15,4//15,-4//15,0,4//15,4//15,0,-4//15,0,0,4//15,-8//15,-8//15,-4//5,-4//15,-4//15,-4//15,0,-8//15,-8//15,-4//5,-4//5,-8//15,-4//15,0,-8//15,-4//5,8//15,0,4//15,4//15,-4//15,-4//15,0,0
        ],
        primitive_beta = BigInt(-7),
        primitive_coefficients = BigInt[
            -2,-1,-1,0,-1,0,0,-1,-1,-1,0,0,0,0,-1,1,-1,0,0,1,0,-1,1,2,1,1,0,0,0,2,1,-1,0,1,1,0,-1,0,0,1,-2,-2,-3,-1,-1,-1,0,-2,-2,-3,-3,-2,-1,0,-2,-3,2,0,1,1,-1,-1,0,0
        ],
        saturation_count_from_search = 606,
    ),
    (
        candidate_id = "F29_pair_1_849",
        source = :ordered_pair,
        pair_j = 1,
        pair_k = 849,
        beta = -28//15,
        coefficients = Rational{BigInt}[
            -8//45,4//45,-4//15,0,-4//15,0,-16//45,-28//45,-4//15,-4//15,16//45,0,16//45,0,4//45,4//15,-4//15,0,0,-4//45,16//45,-4//15,4//15,8//45,4//15,4//15,0,0,0,8//15,4//15,4//45,0,4//15,28//45,0,-4//15,-16//45,0,-4//45,-8//15,-8//15,-4//5,-4//15,-4//15,-4//15,0,-8//45,-8//9,-4//5,-52//45,-8//15,-28//45,0,-8//9,-4//9,8//15,0,4//15,4//15,-4//15,-4//15,0,0
        ],
        primitive_beta = BigInt(-21),
        primitive_coefficients = BigInt[
            -2,1,-3,0,-3,0,-4,-7,-3,-3,4,0,4,0,1,3,-3,0,0,-1,4,-3,3,2,3,3,0,0,0,6,3,1,0,3,7,0,-3,-4,0,-1,-6,-6,-9,-3,-3,-3,0,-2,-10,-9,-13,-6,-7,0,-10,-5,6,0,3,3,-3,-3,0,0
        ],
        saturation_count_from_search = 234,
    )
]

@assert length(RAW_CANDIDATES) == 4
@assert length(unique(candidate.candidate_id for candidate in RAW_CANDIDATES)) == 4
@assert all(length(candidate.coefficients) == 64 for candidate in RAW_CANDIDATES)
@assert all(length(candidate.primitive_coefficients) == 64 for candidate in RAW_CANDIDATES)


## 5. Independent exact audit

For every embedded facet, the notebook verifies exactly:

- validity on all 8128 siglocal vertices;
- minimum slack zero on each orientation and on their union;
- the exact positive white-noise slack $I(u)-\beta$;
- class-29 target slack equal to $-8/45$;
- agreement with the stored saturation count;
- affine dimension 49 for the saturating face.

Only the subsequent SDP calculations use floating-point coefficients.


In [15]:
to_big_rational(value::Rational) =
    BigInt(numerator(value)) // BigInt(denominator(value))

function exact_value(coefficients, behavior::AbstractVector{<:Integer})
    return sum(
        coefficients[index] * BigInt(behavior[index])
        for index in eachindex(coefficients)
    )
end

function exact_value(coefficients, behavior::AbstractVector{<:Rational})
    return sum(
        coefficients[index] * to_big_rational(behavior[index])
        for index in eachindex(coefficients)
    )
end

function exact_vertex_slacks(vertices, coefficients, beta)
    return Rational{BigInt}[
        exact_value(coefficients, collect(@view vertices[row, :])) - beta
        for row in axes(vertices, 1)
    ]
end

function prepare_candidate(raw_candidate)
    uniform_value = exact_value(
        raw_candidate.coefficients,
        UNIFORM_BEHAVIOUR,
    )
    uniform_slack = uniform_value - raw_candidate.beta
    uniform_slack > 0 || error(
        "Candidate $(raw_candidate.candidate_id) has nonpositive uniform slack",
    )
    return merge(raw_candidate, (
        uniform_slack=uniform_slack,
    ))
end

normalized_candidates = prepare_candidate.(RAW_CANDIDATES)
audited_candidates = Any[]

println("\nExact facet audit")
println("candidate          pair       beta      target-slack  saturating  face-dim  facet")

for candidate in normalized_candidates
    coefficients = candidate.coefficients
    beta = candidate.beta

    cycle_slacks = exact_vertex_slacks(vertices_cycle, coefficients, beta)
    clockwise_slacks = exact_vertex_slacks(
        vertices_clockwise,
        coefficients,
        beta,
    )
    counterclockwise_slacks = exact_vertex_slacks(
        vertices_counterclockwise,
        coefficients,
        beta,
    )

    uniform_slack = exact_value(coefficients, UNIFORM_BEHAVIOUR) - beta
    target_slack = exact_value(coefficients, CLASS29_TARGET) - beta

    @assert minimum(cycle_slacks) == 0
    @assert minimum(clockwise_slacks) == 0
    @assert minimum(counterclockwise_slacks) == 0
    @assert all(slack -> slack >= 0, cycle_slacks)
    @assert beta == -BigInt(28)//BigInt(15)
    @assert target_slack == -BigInt(8)//BigInt(45)

    saturating_indices = findall(iszero, cycle_slacks)
    @assert length(saturating_indices) == candidate.saturation_count_from_search
    face_dimension = certified_affine_dimension(
        vertices_cycle[saturating_indices, :],
    )
    is_facet = face_dimension == SIGLOCAL_AFFINE_DIMENSION - 1
    @assert face_dimension == 49
    @assert is_facet

    audited = merge(candidate, (
        uniform_slack=uniform_slack,
        target_slack=target_slack,
        saturation_count=length(saturating_indices),
        face_dimension=face_dimension,
        is_facet=is_facet,
        key=join(string.(candidate.primitive_coefficients), ",") *
            "|beta=" * string(candidate.primitive_beta),
    ))
    push!(audited_candidates, audited)

    pair_label = "($(candidate.pair_j),$(candidate.pair_k))"
    @printf(
        "%-18s %-10s %-9s %-13s %10d %9d  %s\n",
        candidate.candidate_id,
        pair_label,
        string(beta),
        string(target_slack),
        length(saturating_indices),
        face_dimension,
        string(is_facet),
    )
end

@assert length(unique(candidate.key for candidate in audited_candidates)) == 4
println("Exact audit passed for all four distinct facets.")



Exact facet audit
candidate          pair       beta      target-slack  saturating  face-dim  facet
F29_unperturbed    (0,0)      -28//15   -8//45               270        49  true
F29_pair_1_3       (1,3)      -28//15   -8//45               322        49  true
F29_pair_1_585     (1,585)    -28//15   -8//45               606        49  true
F29_pair_1_849     (1,849)    -28//15   -8//45               234        49  true
Exact audit passed for all four distinct facets.


## 6. Bell-operator construction

For each probability coordinate, the corresponding operator is

$$
P^A_{a\mid x}P^B_{b\mid y}P^C_{c\mid z}.
$$

The same `prob_index` map used throughout the notebook places every exact
coefficient in front of the correct operator. No coefficient matrix is
manually reordered.

The compatibility method below addresses the sparse-matrix/`VariableRef`
combination encountered with `QuantumNPA`, `JuMP`, and recent Julia versions.
It is installed only when the active packages do not already provide it.


In [17]:
function install_quantumnpa_jump_compatibility!()
    signature = Tuple{
        SparseMatrixCSC{Float64, Int},
        Symmetric{JuMP.VariableRef, Matrix{JuMP.VariableRef}},
    }

    if !isdefined(QuantumNPA, :dot) ||
       !hasmethod(getfield(QuantumNPA, :dot), signature)
        @eval QuantumNPA begin
            import JuMP: VariableRef
            import LinearAlgebra: Symmetric
            import SparseArrays: SparseMatrixCSC

            function dot(
                A::SparseMatrixCSC{Float64, Int},
                B::Symmetric{VariableRef, Matrix{VariableRef}},
            )
                return sum(
                    A[i, j] * B[j, i]
                    for i in 1:size(A, 1), j in 1:size(A, 2)
                )
            end
        end
        println("Installed the QuantumNPA/JuMP sparse-dot compatibility method.")
    else
        println("QuantumNPA/JuMP sparse-dot compatibility method already available.")
    end
end

install_quantumnpa_jump_compatibility!()

function build_probability_operators()
    PA = projector(1, 1:2, 1:2, full=true)
    PB = projector(2, 1:2, 1:2, full=true)
    PC = projector(3, 1:2, 1:2, full=true)

    operators = Vector{Any}(undef, 64)
    for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
        operators[prob_index(a, b, c, x, y, z)] =
            PA[a + 1, x + 1] * PB[b + 1, y + 1] * PC[c + 1, z + 1]
    end
    return operators
end

function build_bell_expression(coefficients, probability_operators)
    length(coefficients) == length(probability_operators) == 64 ||
        error("A Bell expression requires exactly 64 coefficients/operators")

    expression = nothing
    for index in eachindex(coefficients)
        iszero(coefficients[index]) && continue
        term = Float64(coefficients[index]) * probability_operators[index]
        expression = expression === nothing ? term : expression + term
    end
    expression === nothing && error("The Bell functional is identically zero")
    return expression
end

PROBABILITY_OPERATORS = build_probability_operators()
@assert length(PROBABILITY_OPERATORS) == 64

for candidate in audited_candidates
    build_bell_expression(candidate.coefficients, PROBABILITY_OPERATORS)
end
println("All four Bell expressions were constructed in canonical order.")


QuantumNPA/JuMP sparse-dot compatibility method already available.
All four Bell expressions were constructed in canonical order.


## 7. Restartable NPA scan

The notebook computes levels 2, 3, and 4 for every facet. After every
successful SDP it writes the tables and checkpoint. The saved direct value is

$$
L_k=\min_{p\in\mathcal Q_k}I(p),
$$

and the direct slack is $\Delta_k=L_k-\beta$. Writing
$U=I(u)-\beta>0$, the white-noise-normalized slack is $\Delta_k/U$.
A negative direct slack gives the NPA white-noise quantity

$$
r_k^{\mathrm{NPA}}=\frac{-\Delta_k}{U-\Delta_k}.
$$

This is an optimistic quantity permitted by the outer relaxation, not yet the
robustness of an explicit quantum realization.


In [19]:
function npa_paths(output_directory)
    return (
        checkpoint=joinpath(output_directory, "npa_checkpoint.jls"),
        all_levels=joinpath(output_directory, "class29_npa_all_levels.tsv"),
        level4=joinpath(output_directory, "class29_npa_level4_ranking.tsv"),
        metadata=joinpath(output_directory, "run_metadata.txt"),
    )
end

function write_checkpoint(path, state)
    temporary_path = path * ".tmp"
    open(temporary_path, "w") do io
        serialize(io, state)
    end
    mv(temporary_path, path; force=true)
    return path
end

function candidate_keys(candidates)
    return [candidate.key for candidate in candidates]
end

function new_npa_state(candidates, levels)
    return Dict{String, Any}(
        "format_version" => 1,
        "candidate_keys" => candidate_keys(candidates),
        "levels" => collect(levels),
        "values" => Dict{String, Dict{Int, Float64}}(),
        "seconds" => Dict{String, Dict{Int, Float64}}(),
        "errors" => Dict{String, Dict{Int, String}}(),
        "updated_at" => string(now()),
    )
end

function load_or_create_npa_state(path, candidates, levels; resume=true)
    if resume && isfile(path)
        state = open(path, "r") do io
            deserialize(io)
        end
        state["format_version"] == 1 || error("Unknown NPA checkpoint format")
        state["candidate_keys"] == candidate_keys(candidates) || error(
            "Candidate set differs from the existing checkpoint. " *
            "Use a fresh output directory or set RESUME_NPA_SCAN=false.",
        )
        state["levels"] == collect(levels) || error(
            "NPA_LEVELS differs from the existing checkpoint.",
        )
        println("Resuming NPA scan from: ", abspath(path))
        return state
    end

    println("Starting a new NPA scan.")
    return new_npa_state(candidates, levels)
end

function value_at(state, candidate, level)
    values = get(state["values"], candidate.key, nothing)
    values === nothing && return missing
    return get(values, level, missing)
end

function screening_label(slack; tolerance=NPA_SCREENING_TOLERANCE)
    ismissing(slack) && return "incomplete"
    if slack < -tolerance
        return "npa_candidate"
    elseif slack > tolerance
        return "npa_no_violation"
    else
        return "numerically_unresolved"
    end
end

function npa_noise_quantity(slack, uniform_slack)
    ismissing(slack) && return missing
    return slack < 0 ? -slack / (Float64(uniform_slack) - slack) : 0.0
end

function monotonicity_ok(state, candidate, levels; tolerance)
    values = [value_at(state, candidate, level) for level in levels]
    any(ismissing, values) && return missing
    return all(
        values[index] <= values[index + 1] + tolerance
        for index in 1:(length(values) - 1)
    )
end

text_value(value::Missing) = ""
text_value(value::Bool) = string(value)
text_value(value::Real) = @sprintf("%.16g", Float64(value))
text_value(value) = string(value)

function write_all_levels(path, candidates, state, levels)
    open(path, "w") do io
        println(io, join([
            "candidate_id",
            "source",
            "pair_j",
            "pair_k",
            "level",
            "beta",
            "npa_value",
            "npa_slack",
            "normalized_npa_slack",
            "beta_minus_npa_value",
            "npa_noise_upper_bound",
            "classification",
            "elapsed_seconds",
        ], '\t'))

        for candidate in candidates, level in levels
            value = value_at(state, candidate, level)
            ismissing(value) && continue
            beta = Float64(candidate.beta)
            slack = value - beta
            seconds = get(state["seconds"][candidate.key], level, missing)
            println(io, join([
                candidate.candidate_id,
                string(candidate.source),
                string(candidate.pair_j),
                string(candidate.pair_k),
                string(level),
                text_value(beta),
                text_value(value),
                text_value(slack),
                text_value(slack / Float64(candidate.uniform_slack)),
                text_value(beta - value),
                text_value(npa_noise_quantity(slack, candidate.uniform_slack)),
                screening_label(slack),
                text_value(seconds),
            ], '\t'))
        end
    end
    return path
end

function level4_rows(candidates, state; level=4)
    rows = Any[]
    for candidate in candidates
        value = value_at(state, candidate, level)
        beta = Float64(candidate.beta)
        slack = ismissing(value) ? missing : value - beta
        push!(rows, (
            candidate=candidate,
            value=value,
            slack=slack,
            normalized_slack=ismissing(slack) ? missing :
                slack / Float64(candidate.uniform_slack),
            noise=npa_noise_quantity(slack, candidate.uniform_slack),
            label=screening_label(slack),
            monotonic=monotonicity_ok(
                state,
                candidate,
                NPA_LEVELS;
                tolerance=NPA_MONOTONICITY_TOLERANCE,
            ),
        ))
    end
    sort!(rows; by=row -> ismissing(row.normalized_slack) ? Inf : row.normalized_slack)
    return rows
end

function write_level4_ranking(path, candidates, state)
    rows = level4_rows(candidates, state)
    open(path, "w") do io
        println(io, join([
            "rank",
            "candidate_id",
            "source",
            "pair_j",
            "pair_k",
            "level4_npa_value",
            "beta",
            "npa_slack",
            "normalized_npa_slack",
            "beta_minus_npa_value",
            "npa_noise_upper_bound",
            "npa_monotonicity_ok",
            "classification",
            "saturation_count",
            "face_dimension",
            "primitive_beta",
            "primitive_coefficients_in_canonical_order",
        ], '\t'))

        for (rank_index, row) in enumerate(rows)
            candidate = row.candidate
            beta = Float64(candidate.beta)
            println(io, join([
                string(rank_index),
                candidate.candidate_id,
                string(candidate.source),
                string(candidate.pair_j),
                string(candidate.pair_k),
                text_value(row.value),
                text_value(beta),
                text_value(row.slack),
                text_value(row.normalized_slack),
                ismissing(row.value) ? "" : text_value(beta - row.value),
                text_value(row.noise),
                text_value(row.monotonic),
                row.label,
                string(candidate.saturation_count),
                string(candidate.face_dimension),
                string(candidate.primitive_beta),
                join(string.(candidate.primitive_coefficients), ","),
            ], '\t'))
        end
    end
    return rows
end

function write_result_tables(paths, candidates, state, levels)
    write_all_levels(paths.all_levels, candidates, state, levels)
    return write_level4_ranking(paths.level4, candidates, state)
end


write_result_tables (generic function with 1 method)

In [20]:
function run_npa_scan!(
    candidates,
    probability_operators;
    levels,
    output_directory,
    resume,
    stop_on_error,
)
    mkpath(output_directory)
    paths = npa_paths(output_directory)
    state = load_or_create_npa_state(
        paths.checkpoint,
        candidates,
        levels;
        resume=resume,
    )

    for (candidate_index, candidate) in enumerate(candidates)
        candidate_values = get!(
            state["values"],
            candidate.key,
            Dict{Int, Float64}(),
        )
        candidate_seconds = get!(
            state["seconds"],
            candidate.key,
            Dict{Int, Float64}(),
        )
        candidate_errors = get!(
            state["errors"],
            candidate.key,
            Dict{Int, String}(),
        )

        pending_levels = [
            level for level in levels
            if !haskey(candidate_values, level)
        ]
        if isempty(pending_levels)
            println(
                "[$candidate_index/$(length(candidates))] " *
                "$(candidate.candidate_id): already complete",
            )
            continue
        end

        expression = build_bell_expression(
            candidate.coefficients,
            probability_operators,
        )

        for level in pending_levels
            println(
                "[$candidate_index/$(length(candidates))] " *
                "$(candidate.candidate_id): starting NPA level $level",
            )
            started_at = time()

            try
                bound = npa_min(
                    expression,
                    level;
                    solver=TIGHT_SCS_SOLVER,
                )
                isfinite(bound) || error(
                    "QuantumNPA returned non-finite value $bound",
                )

                candidate_values[level] = Float64(bound)
                candidate_seconds[level] = time() - started_at
                delete!(candidate_errors, level)
                state["updated_at"] = string(now())

                write_checkpoint(paths.checkpoint, state)
                write_result_tables(paths, candidates, state, levels)

                beta = Float64(candidate.beta)
                slack = Float64(bound) - beta
                @printf(
                    "    direct value = %.12f; beta = %.12f; slack = %.12f; elapsed = %.1f s\n",
                    Float64(bound),
                    beta,
                    slack,
                    candidate_seconds[level],
                )
            catch error_value
                candidate_errors[level] = sprint(showerror, error_value)
                state["updated_at"] = string(now())
                write_checkpoint(paths.checkpoint, state)
                write_result_tables(paths, candidates, state, levels)
                @error "NPA calculation failed" candidate=candidate.candidate_id level=level exception=(error_value, catch_backtrace())
                stop_on_error && rethrow()
            end
        end

        expression = nothing
        GC.gc()
    end

    rows = write_result_tables(paths, candidates, state, levels)
    write_checkpoint(paths.checkpoint, state)
    return state, rows, paths
end

npa_state, ranked_rows, output_paths = run_npa_scan!(
    audited_candidates,
    PROBABILITY_OPERATORS;
    levels=NPA_LEVELS,
    output_directory=RESULTS_DIRECTORY,
    resume=RESUME_NPA_SCAN,
    stop_on_error=STOP_ON_NPA_ERROR,
)


Starting a new NPA scan.
[1/4] F29_unperturbed: starting NPA level 2
    direct value = -1.878122754982; beta = -1.866666666667; slack = -0.011456088315; elapsed = 28.0 s
[1/4] F29_unperturbed: starting NPA level 3
    direct value = -1.869626811842; beta = -1.866666666667; slack = -0.002960145176; elapsed = 793.3 s
[1/4] F29_unperturbed: starting NPA level 4
    direct value = -1.869626728528; beta = -1.866666666667; slack = -0.002960061861; elapsed = 78.1 s
[2/4] F29_pair_1_3: starting NPA level 2
    direct value = -1.878428455093; beta = -1.866666666667; slack = -0.011761788427; elapsed = 7.0 s
[2/4] F29_pair_1_3: starting NPA level 3
    direct value = -1.870743798201; beta = -1.866666666667; slack = -0.004077131534; elapsed = 14.2 s
[2/4] F29_pair_1_3: starting NPA level 4
    direct value = -1.870743820390; beta = -1.866666666667; slack = -0.004077153723; elapsed = 54.0 s
[3/4] F29_pair_1_585: starting NPA level 2
    direct value = -1.878243026146; beta = -1.866666666667; slack

(Dict{String, Any}("format_version" => 1, "levels" => [2, 3, 4], "errors" => Dict("-2,1,-3,0,-3,0,-4,-7,-3,-3,4,0,4,0,1,3,-3,0,0,-1,4,-3,3,2,3,3,0,0,0,6,3,1,0,3,7,0,-3,-4,0,-1,-6,-6,-9,-3,-3,-3,0,-2,-10,-9,-13,-6,-7,0,-10,-5,6,0,3,3,-3,-3,0,0|beta=-21" => Dict(), "-2,-1,-1,0,-1,0,0,-1,-1,-1,0,0,0,0,-1,-1,-1,0,0,1,0,-1,1,0,1,1,0,0,0,2,1,1,0,1,1,0,-1,0,0,-1,-2,-2,-3,-1,-1,-1,0,0,-2,-3,-3,-2,-1,0,-2,-1,2,0,1,1,-1,-1,0,0|beta=-7" => Dict(), "-2,-1,-1,0,-1,0,0,-1,-1,-1,0,0,0,0,-1,1,-1,0,0,1,0,-1,1,2,1,1,0,0,0,2,1,-1,0,1,1,0,-1,0,0,1,-2,-2,-3,-1,-1,-1,0,-2,-2,-3,-3,-2,-1,0,-2,-3,2,0,1,1,-1,-1,0,0|beta=-7" => Dict(), "-2,1,-3,0,-3,0,-4,-7,-3,-3,4,0,4,0,1,1,-3,0,0,-1,4,-3,3,0,3,3,0,0,0,6,3,3,0,3,7,0,-3,-4,0,-3,-6,-6,-9,-3,-3,-3,0,0,-10,-9,-13,-6,-7,0,-10,-3,6,0,3,3,-3,-3,0,0|beta=-21" => Dict()), "values" => Dict("-2,1,-3,0,-3,0,-4,-7,-3,-3,4,0,4,0,1,3,-3,0,0,-1,4,-3,3,2,3,3,0,0,0,6,3,1,0,3,7,0,-3,-4,0,-1,-6,-6,-9,-3,-3,-3,0,-2,-10,-9,-13,-6,-7,0,-10,-5,6,0,3,3,-3,-3,0,0|beta=-21" => Dict(4 =>

## 8. Level-4 ranking and numerical diagnostics

Candidates are ranked by the white-noise-normalized slack
$(L_4-\beta)/(I(u)-\beta)$. Thus the most negative entry corresponds to the
largest optimistic white-noise tolerance at level 4.

The monotonicity check verifies the expected minimization ordering
$L_2\leq L_3\leq L_4$ within the configured numerical tolerance.


In [22]:
ranked_rows = level4_rows(audited_candidates, npa_state)

println("\nLevel-4 NPA results ranked by normalized slack")
println("rank  candidate          pair       L2            L3            L4            beta          L4-beta       norm-slack    noise         label")

for (rank_index, row) in enumerate(ranked_rows)
    candidate = row.candidate
    l2 = value_at(npa_state, candidate, 2)
    l3 = value_at(npa_state, candidate, 3)
    l4 = value_at(npa_state, candidate, 4)
    pair_label = "($(candidate.pair_j),$(candidate.pair_k))"

    @printf(
        "%4d  %-18s %-10s %-13s %-13s %-13s %-13.9f %-13s %-13s %-13s %s\n",
        rank_index,
        candidate.candidate_id,
        pair_label,
        ismissing(l2) ? "—" : @sprintf("%.9f", l2),
        ismissing(l3) ? "—" : @sprintf("%.9f", l3),
        ismissing(l4) ? "—" : @sprintf("%.9f", l4),
        Float64(candidate.beta),
        ismissing(row.slack) ? "—" : @sprintf("%.9f", row.slack),
        ismissing(row.normalized_slack) ? "—" : @sprintf("%.9f", row.normalized_slack),
        ismissing(row.noise) ? "—" : @sprintf("%.9f", row.noise),
        row.label,
    )

    if row.monotonic === false
        @warn "NPA levels are not numerically monotone within tolerance" candidate=candidate.candidate_id level2=l2 level3=l3 level4=l4
    end

    if !ismissing(l4) &&
       l4 > Float64(candidate.beta) + NPA_MONOTONICITY_TOLERANCE
        @warn "Level-4 lower bound exceeds the exact siglocal bound" candidate=candidate.candidate_id level4=l4 beta=candidate.beta
    end
end

promising_count = count(row -> row.label == "npa_candidate", ranked_rows)
println("\nPromising level-4 candidates: ", promising_count)
println("All-level table: ", abspath(output_paths.all_levels))
println("Level-4 table:   ", abspath(output_paths.level4))
println("Checkpoint:      ", abspath(output_paths.checkpoint))



Level-4 NPA results ranked by normalized slack
rank  candidate          pair       L2            L3            L4            beta          L4-beta       norm-slack    noise         label
   1  F29_pair_1_3       (1,3)      -1.878428455  -1.870743798  -1.870743820  -1.866666667  -0.004077154  -0.004077154  0.004060598   npa_candidate
   2  F29_unperturbed    (0,0)      -1.878122755  -1.869626812  -1.869626729  -1.866666667  -0.002960062  -0.002960062  0.002951326   npa_candidate
   3  F29_pair_1_585     (1,585)    -1.878243026  -1.868895169  -1.868894472  -1.866666667  -0.002227805  -0.002227805  0.002222853   npa_candidate
   4  F29_pair_1_849     (1,849)    -1.878109748  -1.868836640  -1.868828512  -1.866666667  -0.002161845  -0.002161845  0.002157181   npa_candidate

Promising level-4 candidates: 4
All-level table: C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_project\CV_dropbox_content\Projeto_Ze_Gabrielzinho_Rabelo_Planeta\Jupiter\cycle_class2

## 9. Reproducibility metadata

The final cell records Julia and package versions, solver settings, candidate
provenance, and the exact primitive inequalities. After the run, send the two
result TSV files, `run_metadata.txt`, and the executed notebook for validation.


In [24]:
function installed_version(package_name)
    for package in values(Pkg.dependencies())
        if package.name == package_name
            return package.version === nothing ? "unknown" : string(package.version)
        end
    end
    return "not found"
end

open(output_paths.metadata, "w") do io
    println(io, "Class-29 perturbed-facet NPA screening")
    println(io, "updated_at = ", npa_state["updated_at"])
    println(io, "julia_version = ", VERSION)
    println(io, "julia_threads = ", Threads.nthreads())
    for package in required_packages
        println(io, lowercase(package), "_version = ", installed_version(package))
    end
    println(io, "candidate_count = ", length(audited_candidates))
    println(io, "npa_levels = ", join(NPA_LEVELS, ","))
    println(io, "scs_eps_abs = ", SCS_EPS_ABS)
    println(io, "scs_eps_rel = ", SCS_EPS_REL)
    println(io, "scs_max_iters = ", SCS_MAX_ITERS)
    println(io, "screening_tolerance = ", NPA_SCREENING_TOLERANCE)
    println(io, "monotonicity_tolerance = ", NPA_MONOTONICITY_TOLERANCE)
    println(io, "coordinate_order = xyz blocks lexicographic; abc inside each block lexicographic")
    println(io, "siglocal_vertex_count = ", size(vertices_cycle, 1))
    println(io, "siglocal_affine_dimension = ", SIGLOCAL_AFFINE_DIMENSION)
    println(io, "discovery_epsilons = 1/1000,1/100")
    println(io, "epsilon_runs_returned_same_exact_facets = true")

    for candidate in audited_candidates
        prefix = candidate.candidate_id
        println(io, prefix, "_source = ", candidate.source)
        println(io, prefix, "_pair = (", candidate.pair_j, ",", candidate.pair_k, ")")
        println(io, prefix, "_normalized_beta = ", candidate.beta)
        println(io, prefix, "_primitive_beta = ", candidate.primitive_beta)
        println(io, prefix, "_primitive_coefficients = ", join(candidate.primitive_coefficients, ","))
        println(io, prefix, "_saturation_count = ", candidate.saturation_count)
        println(io, prefix, "_face_dimension = ", candidate.face_dimension)
    end
end

println("Metadata: ", abspath(output_paths.metadata))
println("\nInstalled package versions")
for package in required_packages
    println("  ", rpad(package, 20), installed_version(package))
end


Metadata: C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_project\CV_dropbox_content\Projeto_Ze_Gabrielzinho_Rabelo_Planeta\Jupiter\cycle_class29_perturbed_npa_results\run_metadata.txt

Installed package versions
  JuMP                1.30.1
  MathOptInterface    1.54.0
  QuantumNPA          0.1.2
  SCS                 2.7.0


## What to send back

After the run finishes, send:

1. `class29_npa_all_levels.tsv`;
2. `class29_npa_level4_ranking.tsv`;
3. `run_metadata.txt`;
4. this executed notebook.

The binary checkpoint is needed only if the scan must be resumed locally. If
one or more facets have a clearly negative level-4 slack, the next step is a
dedicated see-saw optimization for each promising facet.
